<a href="https://colab.research.google.com/github/tqd3pz/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [28]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [29]:
pop_tarts = cat[cat["product_name"].str.lower().str.contains("pop")]

print("Number of Pop-Tart rows:", len(pop_tarts))
print(pop_tarts)

Number of Pop-Tart rows: 3
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [30]:
strawberry = pop_tarts[pop_tarts["product_name"].str.lower().str.contains("strawberry")]

print(strawberry[["sku", "product_name"]])

             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry


**Pop-Tart rows that are a different product:** _..._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [31]:
sku_mapping = {
    "PT-12": "PT-12",
    "SKU#459812": "PT-12",
    "POPTART-STRAW": "PT-12"
}

cat["canonical_sku"] = cat["sku"].replace(sku_mapping)

print(cat[["sku", "canonical_sku", "product_name"]])

             sku canonical_sku          product_name
0          PT-12         PT-12  pop-tarts strawberry
1     SKU#459812         PT-12   Strawberry Pop-Tart
2  POPTART-STRAW         PT-12  Pop-Tarts Strawberry


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [32]:
print("Before:")
print(cat["category"].value_counts())

cat["category"] = cat["category"].str.strip().str.title()

cat["category"] = cat["category"].replace("Breakfast Foods", "Breakfast")

print("After:")
print(cat["category"].value_counts())

Before:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64
After:
category
Breakfast    3
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [33]:
def normalize_name(name):
    name = name.lower()
    name = "".join(char for char in name if char.isalnum() or char.isspace())
    words = name.split()
    words.sort()
    return " ".join(words)

cat["normalized_name"] = cat["product_name"].apply(normalize_name)

groups = cat.groupby("normalized_name")["sku"].nunique()

duplicates = groups[groups > 1]

print(duplicates)

for name in duplicates.index:
    print(cat[cat["normalized_name"] == name][["sku", "product_name"]])

normalized_name
poptarts strawberry    2
Name: sku, dtype: int64
             sku          product_name
0          PT-12  pop-tarts strawberry
2  POPTART-STRAW  Pop-Tarts Strawberry


In [34]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'sample_data']


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [35]:
assert cat["canonical_sku"].isnull().sum() == 0

assert len(cat) == len(cat["canonical_sku"])

print("Mapping is safe!")

Mapping is safe!


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Based on the data currently loaded, I identified the strawberry Pop-Tart family as fragmented. The different SKUs represent the same product but have variations in spelling and formatting. I would group them under one canonical SKU.
b)Consolidation becomes a business decision when two products have similar names but may not actually be the same product. For example, I would be unsure whether Pop-Tarts with different flavors should be grouped under one product family or kept separate. I would ask the category manager or the person responsible for the product catalog to clarify how these products should be classified before combining them.